# 04 - Scheduling: deterministic baseline vs QpiAI Opt

**Purpose:** Solve the wave schedule exactly with the deterministic baseline, check the same MILP with QpiAI Opt's offline tools, and visualise the waves and constraint use.

**Inputs:** Phase 2 problem; qpiai-opt (offline classify / inspect / feasibility only)

**Outputs:** wave timeline chart, capacity/window chart, Opt model checks

**Status labels:** Baseline VERIFIED. QpiAI Opt offline checks VERIFIED. Authenticated `solve(mode='local')` VERIFIED once (owner-approved 2026-10-29): optimal, same objective and assignment as the baseline (not better). No platform, submit() or QPU.

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
from cryptoshift.planning.planner import build_problem, schedule, to_tasks
p = build_problem(); s = schedule(p); c = s['config']
print('feasible', s['feasible'], '| objective', s['objective'], '| dependency violations', len(s['dependency_violations']), '| constraint violations', len(s['constraint_violations']))
tasks = pd.DataFrame(s['tasks']).set_index('service_id')
tasks['wave'] = pd.Series(s['task_to_wave'])
tasks[['wave','duration_minutes','risk_score','verification_ready_wave0','dependencies']].sort_values('wave')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [2, 1]})
pal = plt.get_cmap('tab10')
for w in sorted(tasks['wave'].unique()):
    start = 0
    for i, (sid, r) in enumerate(tasks[tasks['wave']==w].iterrows()):
        ax[0].barh(w, r['duration_minutes'], left=start, color=pal(list(tasks.index).index(sid)), edgecolor='white')
        ax[0].text(start + r['duration_minutes']/2, w, f"{sid}\n{r['duration_minutes']} min", ha='center', va='center', fontsize=8, color='white')
        start += r['duration_minutes']
ax[0].axvline(c['max_minutes_per_wave'], color='red', ls='--', label=f"window {c['max_minutes_per_wave']} min (ASSUMPTION)")
ax[0].set_yticks(sorted(tasks['wave'].unique()), [f'Wave {w}' for w in sorted(tasks['wave'].unique())]); ax[0].invert_yaxis(); ax[0].set_xlabel('minutes'); ax[0].legend(); ax[0].set_title('Migration waves (deterministic exact baseline)')
use = tasks.groupby('wave').agg(services=('duration_minutes','size'), minutes=('duration_minutes','sum'))
(use / [c['max_services_per_wave'], c['max_minutes_per_wave']]).plot(kind='bar', ax=ax[1], color=['#457b9d','#e9c46a'], title='Constraint utilisation per wave')
ax[1].axhline(1, color='red', ls='--'); ax[1].set_ylabel('fraction of limit'); ax[1].set_ylim(0, 1.2)
savefig('04_waves')

In [ ]:
from cryptoshift.optimization.member3_qpiai import opt_wave_model_checks
tl, _ = to_tasks(p)
o = opt_wave_model_checks(tl, c, s['task_to_wave'])
pd.Series({k: o[k] for k in ('sdk_versions','problem_class','binary_variables','constraints','baseline_feasible_in_opt_model','baseline_objective_recomputed')} | {'authenticated_local_solve (recorded owner-approved run)': (lambda r: f"{r['status']}: {r.get('solver_status', r.get('reason', ''))}, objective {r.get('objective')} vs baseline {r.get('same_input_comparison', {}).get('baseline_objective')} (same assignment: {r.get('same_input_comparison', {}).get('same_assignment')})")(json.loads((DERIVED / 'scheduling_comparison.json').read_text())['qpiai_opt']['authenticated_local_solve'])}).to_frame('QpiAI Opt')

**Plain language.** Each service gets exactly one wave; a provider must move no later than its consumers; each wave holds at most 2 services and 85 minutes (assumptions). The baseline tries every assignment and keeps the cheapest valid one, so its answer is optimal for this small problem. QpiAI Opt accepted the same model as a MILP and confirmed the baseline answer is feasible in it. Solving with QpiAI needs an API key and owner authorization, which this run did not have, so there is no QpiAI-vs-baseline objective comparison yet.